# Compare ancestry keep-lists

Discovers every keep-list this pipeline's ancestry-filtering stages have produced for a `CDR_VERSION` -- the premade-label base cohorts (`01_premade_label_filter.ipynb`, one per `BASE_GROUP`: `eur`, `afr`) and the final, 1000G-referenced-Mahalanobis-filtered sample sets (`05_round2_1000g_filter.ipynb`, one per `SAMPLE_SET`: `eur`, `eur_stringent`, `eur_loose`, `eur_premade_label`, `afr`) -- and reports pairwise overlap (raw intersection count and Jaccard index) across all of them, plus same-`BASE_GROUP` retention from the premade base cohort to each final sample set.

Useful for two different questions:
- **Within one `BASE_GROUP`, base -> final**: how much does round 2's 1000G-referenced Mahalanobis filter actually remove, at each stringency? (`final ⊆ premade_base` always, since every `SAMPLE_SET` sharing a `BASE_GROUP` is fit and filtered from that same base cohort -- a strict attrition chain, not just correlation.)
- **Across different `SAMPLE_SET`s**: how much do different ancestry definitions actually agree? E.g. `eur`'s and `eur_premade_label`'s final keep-lists are two different points on the same 1000G-referenced ellipsoid sweep (default threshold vs. no filtering at all) -- comparing them shows how much the ellipsoid step actually changes membership relative to the raw premade label.

No PCA/plink here -- pure ID-set arithmetic over already-written keep-lists. Run this after whichever base groups/sample sets you want compared have already been produced.</cell id="cmp-intro">

## Inputs

In [ ]:
import os
import re
import glob
from itertools import combinations

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

WORKSPACE_BUCKET = os.path.expanduser(
    "~/workspace/Data from All of Us Controlled Tier /shared-env-pilot"
)

# Must match whatever CDR_VERSION the ancestry-filtering notebooks were actually
# run with -- this notebook only reads their output, it doesn't build anything.
CDR_VERSION = "v9"

# Top-level bucket folder name for this project's outputs -- distinct from
# CDR_VERSION, which keeps its real meaning elsewhere. Fixed literal, matches
# every other notebook in this pipeline.
PROJECT_DIR = "covariance_v9"

BUCKET_DIR = f"{WORKSPACE_BUCKET}/{PROJECT_DIR}/01_ancestry_filtering"
OUT_DIR = f"{BUCKET_DIR}/keep_list_comparison_{CDR_VERSION}"
os.makedirs(OUT_DIR, exist_ok=True)

print(BUCKET_DIR)
print(OUT_DIR)

## Discover keep-lists

Globs for every keep-list file matching each stage's own naming convention, rather than hardcoding an expected file list -- a `BASE_GROUP`/`SAMPLE_SET` that hasn't been run yet is just silently absent, not an error. `list_id` is a short human-readable label (`premade_base/eur`, `final/afr`, `final/eur_premade_label`, ...) used throughout the rest of this notebook.</cell id="cmp-discover-md">

In [ ]:
# SAMPLE_SET -> base_group, mirroring 06_final_pca.ipynb's own SAMPLE_SETS dict --
# needed here to compute base->final retention per sample set below
SAMPLE_SET_BASE_GROUP = {
    "eur": "eur",
    "eur_stringent": "eur",
    "eur_loose": "eur",
    "eur_premade_label": "eur",
    "afr": "afr",
}

lists = []   # each: {"list_id", "stage", "sample_set" (or base_group for premade_base), "path"}

for path in sorted(glob.glob(f"{BUCKET_DIR}/premade_label_{CDR_VERSION}/premade_keep_ids_*.txt")):
    m = re.search(r"premade_keep_ids_(.+)\.txt$", os.path.basename(path))
    base_group = m.group(1) if m else "unknown"
    lists.append({"list_id": f"premade_base/{base_group}", "stage": "premade_base", "sample_set": base_group, "path": path})

for path in sorted(glob.glob(f"{BUCKET_DIR}/genome_wide_panel_*/final_pca/final_keep_ids_*.txt")):
    m = re.search(r"final_keep_ids_(.+)_(p[\d.]+|unfiltered)\.txt$", os.path.basename(path))
    sample_set = m.group(1) if m else "unknown"
    lists.append({"list_id": f"final/{sample_set}", "stage": "final", "sample_set": sample_set, "path": path})

print(f"{len(lists)} keep-lists found")
for l in lists:
    print(" ", l["list_id"], "->", l["path"])

## Load ID sets

Same plink-style parsing every other notebook in this pipeline uses (one ID per line, or "FID IID" -- take the last whitespace-separated field either way).

In [ ]:
def load_ids(path):
    with open(path) as f:
        lines = [l.strip() for l in f if l.strip()]
    return set(l.split()[-1] for l in lines)

for l in lists:
    l["ids"] = load_ids(l["path"])
    l["n"] = len(l["ids"])

summary = pd.DataFrame(lists)[["list_id", "stage", "sample_set", "n", "path"]].sort_values(["stage", "sample_set"])
summary_path = os.path.join(OUT_DIR, "keep_list_sizes.tsv")
summary.to_csv(summary_path, sep="\t", index=False)
print(f"Wrote {summary_path}")
summary

## Pairwise overlap matrix

`overlap[i][j]` = `|ids_i ∩ ids_j|`; `jaccard[i][j]` = `|ids_i ∩ ids_j| / |ids_i ∪ ids_j|`. Both are symmetric with the diagonal equal to each list's own size (overlap) or `1.0` (Jaccard). Useful both within a `BASE_GROUP` (premade-base -> final attrition -- see next section for that specifically) and across `SAMPLE_SET`s (how much do independently-thresholded ancestry definitions actually agree).</cell id="cmp-overlap-md">

In [ ]:
n_lists = len(lists)
overlap = np.zeros((n_lists, n_lists), dtype=int)
jaccard = np.zeros((n_lists, n_lists))

for i, j in combinations(range(n_lists), 2):
    inter = len(lists[i]["ids"] & lists[j]["ids"])
    union = len(lists[i]["ids"] | lists[j]["ids"])
    overlap[i, j] = overlap[j, i] = inter
    jaccard[i, j] = jaccard[j, i] = (inter / union if union else 0.0)
for i in range(n_lists):
    overlap[i, i] = lists[i]["n"]
    jaccard[i, i] = 1.0

labels = [l["list_id"] for l in lists]
overlap_df = pd.DataFrame(overlap, index=labels, columns=labels)
jaccard_df = pd.DataFrame(jaccard, index=labels, columns=labels)

overlap_path = os.path.join(OUT_DIR, "keep_list_overlap_counts.tsv")
jaccard_path = os.path.join(OUT_DIR, "keep_list_jaccard.tsv")
overlap_df.to_csv(overlap_path, sep="\t")
jaccard_df.to_csv(jaccard_path, sep="\t")
print(f"Wrote {overlap_path}")
print(f"Wrote {jaccard_path}")

overlap_df

In [ ]:
fig, ax = plt.subplots(figsize=(0.5 * n_lists + 2, 0.5 * n_lists + 2))
im = ax.imshow(jaccard_df.values, cmap="viridis", vmin=0, vmax=1)
ax.set_xticks(range(n_lists))
ax.set_yticks(range(n_lists))
ax.set_xticklabels(labels, rotation=90, fontsize=7)
ax.set_yticklabels(labels, fontsize=7)
for i in range(n_lists):
    for j in range(n_lists):
        ax.text(j, i, f"{jaccard_df.values[i, j]:.2f}", ha="center", va="center",
                color="white" if jaccard_df.values[i, j] < 0.5 else "black", fontsize=6)
fig.colorbar(im, ax=ax, label="Jaccard index")
ax.set_title(f"Keep-list overlap (Jaccard) -- {CDR_VERSION}")
plt.tight_layout()
plot_path = os.path.join(OUT_DIR, f"keep_list_jaccard_heatmap_{CDR_VERSION}.png")
plt.savefig(plot_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved {plot_path}")

## Base-cohort -> final retention

Every final `SAMPLE_SET` is filtered from its `BASE_GROUP`'s own premade-label cohort, so `final ⊆ premade_base` should hold exactly -- `n_not_in_base` below should read 0 for every row; a nonzero value means something's inconsistent (e.g. `BASE_GROUP`/`CDR_VERSION` mismatch between `01_premade_label_filter.ipynb`, `05_round2_1000g_filter.ipynb`, and `06_final_pca.ipynb`) and is worth investigating before trusting the retention percentage next to it. Retention (`n / base_n`) is the fraction of the premade-label cohort that survives round 2's Mahalanobis filter -- low retention isn't necessarily wrong (a tight threshold, e.g. `eur_stringent`, is *supposed* to cut hard; `eur_premade_label` should read exactly 100%, since it skips the ellipsoid entirely), but is worth comparing side by side across sample sets.</cell id="cmp-chain-md">

In [ ]:
by_id = {l["list_id"]: l for l in lists}

rows = []
for sample_set, base_group in SAMPLE_SET_BASE_GROUP.items():
    base = by_id.get(f"premade_base/{base_group}")
    final = by_id.get(f"final/{sample_set}")
    if base is None or final is None:
        continue
    not_in_base = final["ids"] - base["ids"]
    rows.append({
        "sample_set": sample_set,
        "base_group": base_group,
        "base_n": base["n"],
        "final_n": final["n"],
        "retention_pct": round(100 * final["n"] / base["n"], 1) if base["n"] else None,
        "n_not_in_base": len(not_in_base),
    })

retention_df = pd.DataFrame(rows)
retention_path = os.path.join(OUT_DIR, "keep_list_retention_chain.tsv")
retention_df.to_csv(retention_path, sep="\t", index=False)
print(f"Wrote {retention_path}")
retention_df

## Next steps

`keep_list_jaccard.tsv`/the heatmap are the quickest way to spot two things worth following up on:
- **Low Jaccard between `final/eur` and `final/eur_premade_label`** -- shows how much round 2's Mahalanobis filter actually changes membership relative to trusting AoU's premade label outright; a very low overlap would mean the ellipsoid is excluding a large, non-random fraction of the premade-label cohort, worth a closer look at why.
- **`n_not_in_base` nonzero anywhere** in the retention table -- a path/`BASE_GROUP`/`CDR_VERSION` mismatch between `01_premade_label_filter.ipynb`, `05_round2_1000g_filter.ipynb`, and `06_final_pca.ipynb`, not a real biological finding.</cell id="cmp-next-md">